# Sentiment Analysis (Positive / Negative / Neutral)

This notebook trains a TF-IDF + classical ML sentiment model.

## 1. Imports

In [22]:
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


## 2. Load dataset

In [23]:
df = pd.read_csv("demo_sentiment_data.csv")
df.head()

,text,label
0,Poor quality and not worth the money.,negative
1,Support ignored my problem for days.,negative
2,The software was updated last week.,neutral
3,The movie was engaging and well made.,positive
4,Worst experience I have had in a long time.,negative


## 3. Text preprocessing

In [24]:
try:
    import nltk
    from nltk.stem import WordNetLemmatizer, SnowballStemmer
    from nltk.corpus import stopwords as nltk_stopwords
    stop_words = set(nltk_stopwords.words('english'))
except Exception:
    WordNetLemmatizer = None
    SnowballStemmer = None
    stop_words = set(ENGLISH_STOP_WORDS)

class Preprocessor:
    def __init__(self):
        self.lemmatizer = None
        self.stemmer = None
        try:
            self.lemmatizer = WordNetLemmatizer()
            _ = self.lemmatizer.lemmatize("cars")
        except Exception:
            self.lemmatizer = None
        try:
            self.stemmer = SnowballStemmer("english")
        except Exception:
            self.stemmer = None

    def clean(self, text):
        text = str(text)
        text = re.sub(r"[^a-zA-Z\s]", " ", text).lower()
        tokens = []
        for token in text.split():
            if token in stop_words or len(token) < 2:
                continue
            if self.lemmatizer is not None:
                try:
                    token = self.lemmatizer.lemmatize(token)
                except Exception:
                    pass
            elif self.stemmer is not None:
                token = self.stemmer.stem(token)
            tokens.append(token)
        return " ".join(tokens)

preprocessor = Preprocessor()
df["clean_text"] = df["text"].apply(preprocessor.clean)
df[["text", "clean_text", "label"]].head()

,text,clean_text,label
0,Poor quality and not worth the money.,poor quality worth money,negative
1,Support ignored my problem for days.,support ignored problem days,negative
2,The software was updated last week.,software updated week,neutral
3,The movie was engaging and well made.,movie engaging,positive
4,Worst experience I have had in a long time.,worst experience long time,negative


## 4. Train-test split

In [25]:
print(df["label"].value_counts())

label
negative    15
neutral     15
positive    15
Name: count, dtype: int64


## 5. Train two models and compare

In [26]:
X = df["text"].astype(str).tolist()
y = df["label"].astype(str).tolist()
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

Train samples: 36
Test samples: 9


## 6. Evaluate best 

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(max_iter=2000))
    ]),
    "SVM": Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LinearSVC())
    ])
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        preds,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results)

print(results_df)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.111111   0.333333  0.111111  0.166667
1                  SVM  0.111111   0.333333  0.111111  0.166667


## 7. Predict new text

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# Create and train the model
best_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC())
])

best_model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


## 8. Save the model

In [29]:
joblib.dump(best_model, "sentiment_model.joblib")
print("Saved: sentiment_model.joblib")

Saved: sentiment_model.joblib


## 9.Traning the model with real-world examples 

In [30]:
sample_reviews = [
    "This product is amazing and works perfectly.",
    "I love the quality and design.",
    "Excellent customer service and fast delivery.",
    "The food was delicious and fresh.",
    "Very satisfied with my purchase.",
    "This is the worst product I have ever bought.",
    "Terrible experience and poor quality.",
    "The service was very slow and disappointing.",
    "I hate this item and want a refund.",
    "Completely useless and waste of money.",
    "The product is okay.",
    "It works as expected.",
    "Nothing special about this service.",
    "Average quality and reasonable price.",
    "The experience was neither good nor bad."
]

print("===== Sentiment Predictions =====")

for i, review in enumerate(sample_reviews, start=1):
    prediction = best_model.predict([review])[0]
    print(f"{i}. Review: {review}")
    print(f"   Predicted Sentiment: {prediction}")
    print("-" * 50)

===== Sentiment Predictions =====
1. Review: This product is amazing and works perfectly.
   Predicted Sentiment: positive
--------------------------------------------------
2. Review: I love the quality and design.
   Predicted Sentiment: positive
--------------------------------------------------
3. Review: Excellent customer service and fast delivery.
   Predicted Sentiment: positive
--------------------------------------------------
4. Review: The food was delicious and fresh.
   Predicted Sentiment: positive
--------------------------------------------------
5. Review: Very satisfied with my purchase.
   Predicted Sentiment: negative
--------------------------------------------------
6. Review: This is the worst product I have ever bought.
   Predicted Sentiment: negative
--------------------------------------------------
7. Review: Terrible experience and poor quality.
   Predicted Sentiment: negative
--------------------------------------------------
8. Review: The service was v

## 10. user input examples:

In [31]:
# User enters 5 reviews

print("Enter 5 reviews for sentiment analysis:\n")

for i in range(5):
    review = input(f"Review {i+1}: ")
    prediction = best_model.predict([review])[0]

    print(f"Predicted Sentiment: {prediction}")
    print("-" * 40)

Enter 5 reviews for sentiment analysis:



Review 1:  The food was excellent and tasty


Predicted Sentiment: positive
----------------------------------------


Review 2:   The product is okay


Predicted Sentiment: neutral
----------------------------------------


...

Predicted Sentiment: positive
----------------------------------------


Review 4:  I love the quality and design


Predicted Sentiment: positive
----------------------------------------


Review 5:  It works as expected


Predicted Sentiment: positive
----------------------------------------
